# Lecture 8


## Seq2Seq Bottleneck and Attention in RNNs

Standard seq2seq RNNs squash the whole input sequence into one fixed vector $c$ (usually the last hidden state $h_T$). For long sequences, this creates a bottleneck because the model has to cram everything into that single vector.

Attention fixes this by making a new context vector $c_t$ at every decoder step $t$. Instead of relying on just the last hidden state, the decoder looks at all encoder hidden states $h_i$ directly, picking out relevant source tokens as it generates each output token.

Alignment scores $e_{t,i}$ check how well the previous decoder state $s_{t-1}$ matches each encoder state $h_i$. Taking the softmax of these scores gives normalized weights $a_{t,i}$. Then $c_t$ is just the weighted sum of encoder states:
$$c_t = \sum_{i} a_{t,i} h_i$$

The whole setup is fully differentiable, so backprop trains it end to end without needing explicit alignment labels. Since a weighted sum ignores order, attention treats inputs as an unordered set unless you add positional information.


## Image Captioning with Attention

Instead of squeezing an entire image into one global vector, spatial attention takes an $H \times W \times D$ feature map from a CNN. Each grid position $(i, j)$ gives a feature vector $z_{i,j}$ for that specific patch of the image.

At each step $t$, the decoder RNN compares its hidden state against all $H \times W$ spatial features to compute alignment scores. The context vector $c_t$ then focuses on the image regions that matter for whatever word it is generating.


question to revisit: what happens here when the input size isn't divisible?


this derivation felt shaky, redo it on paper before moving on


connection: this shows up again in the architectures lecture, remember it


got tripped up by the stride math here, worth a second pass


intuition check: explain this out loud without looking at the slides


the diagram for this one in the slides made it click finally


reminder: run the code cell above and tweak the filter size


kept confusing these two terms, write them out explicitly next time


skimmed this too fast the first time, the math is simpler than it looked


side note: tried rebuilding this part from memory and mixed up the indexing


## General Attention Framework (Query, Key, Value)

General attention frames everything in terms of Queries ($Q$), Keys ($K$), and Values ($V$). Think of Queries as search vectors, Keys as index vectors for matching, and Values as the content you actually extract.

Dot-product alignment replaces learned linear layers with simple vector dot products. Dividing by $\sqrt{d_k}$ keeps dot products from blowing up in higher dimensions, which stops softmax from saturating and causing vanishing gradients:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$


In [ ]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V)

this derivation felt shaky, redo it on paper before moving on
